# Общая часть практики: от PDF статьи к графу и инструментам агента

Блокнот показывает общую часть практики — пакет `mathkg` — на одной статье arXiv по уравнениям с
частными производными: W. Cintra и др. «Maximum Principles and Principal Eigenvalue for Coupled
Local/Nonlocal Models» (arXiv:2611.16131, 27 страниц). Статья проходит тот же путь, который на
практике проходит каждая из 1000 статей набора студента:

1. общая основа `base` превращает PDF в текст с разметкой (абзацы, предложения, шрифт, формулы,
   гиперссылки PDF);
2. десять модулей извлечения находят в тексте метаданные, разделы, утверждения, определения,
   формулы, литературу, доказательства, ссылки, понятия и предметные области;
3. прогон по набору, проверка контракта и ручной эталон оценивают модули;
4. результаты модулей переводятся в граф RDF, к графу задаются запросы SPARQL;
5. модули становятся инструментами агента;
6. студенческий модуль заменяет базовую версию;
7. общий кэш моделей Hugging Face превращает фрагменты в векторы для поиска.

В Google Colab блокнот выполняется без GPU за 3–5 минут; большая часть времени уходит на установку
библиотек и загрузку модели `e5` в разделе 9. Описание пакета — `ОБЩАЯ_ЧАСТЬ.md` в корне
репозитория.

## 0. Подключение

В Colab ячейка клонирует репозиторий группы и ставит библиотеки из `requirements.txt`; при запуске
на своём компьютере блокнот берёт пакет из папки репозитория, в которой лежит. Все результаты
блокнота пишутся в папку `work/` репозитория, в git она не попадает.

In [1]:
import os, sys, json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO_URL = 'https://github.com/iroli/MUIV3PR.git'   # адрес репозитория группы
    ROOT = '/content/kg-practice-2026'
    if not os.path.exists(ROOT):
        !git clone -q {REPO_URL} {ROOT}
    !pip install -q -r {ROOT}/requirements.txt
else:
    ROOT = os.path.abspath('..')                   # блокнот лежит в папке notebooks/ репозитория
sys.path.insert(0, ROOT)
WORK = os.path.join(ROOT, 'work')                 # рабочая папка блокнота
ONTOLOGY = [os.path.join(ROOT, 'ontology', n) for n in ('MocassinPlus_v2.ttl', 'SciLibReco.ttl')]
os.makedirs(WORK, exist_ok=True)

import pandas as pd
pd.set_option('display.max_colwidth', 110)
print('репозиторий:', ROOT)

репозиторий: /content/kg-practice-2026


## 1. Набор статей

Набор студента — папка с PDF и таблица `index.csv` с полями `doc_id` (уникальный номер статьи во
всей группе) и `file`. Здесь набор состоит из одной статьи arXiv: ячейка скачивает её в
`work/pdf/A0001.pdf`. Свои PDF можно положить в ту же папку: ячейка внесёт в таблицу все файлы
папки под номерами U0001, U0002, …

In [2]:
import urllib.request, csv, glob
PDF_DIR = os.path.join(WORK, 'pdf')
os.makedirs(PDF_DIR, exist_ok=True)
ARXIV_ID = '2611.16131v1'
pdf_path = os.path.join(PDF_DIR, 'A0001.pdf')
if not os.path.exists(pdf_path):
    req = urllib.request.Request(f'https://arxiv.org/pdf/{ARXIV_ID}', headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as r, open(pdf_path, 'wb') as f:
        f.write(r.read())

own = sorted(os.path.basename(p) for p in glob.glob(PDF_DIR + '/*.pdf') if not p.endswith('A0001.pdf'))
rows = [('A0001', 'A0001.pdf')] + [(f'U{k:04d}', name) for k, name in enumerate(own, 1)]
with open(os.path.join(WORK, 'index.csv'), 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows([('doc_id', 'file')] + rows)
pd.DataFrame(rows, columns=['doc_id', 'file'])

,doc_id,file
0,A0001,A0001.pdf


## 2. Общая основа base: PDF → Doc

`load_doc` читает PDF библиотекой PyMuPDF и собирает объект `Doc` — статью, с которой работают все
модули. Текст статьи `doc.text` — одна строка: колонки расставлены по порядку, колонтитулы и номера
страниц убраны, переносы склеены, сноски вынесены в `doc.footnotes`. Всё остальное в `Doc` — отрезки
этой строки (начало и конец в символах): страницы, абзацы, предложения, отметки шрифта, выключные
формулы, гиперссылки PDF. Поэтому позиции фрагментов у всех модулей и у всех студентов совпадают.

`check_text_layer` отсеивает сканы без текстового слоя и PDF со сломанной кодировкой шрифтов: такие
статьи этап 1 исключает из набора.

In [3]:
from mathkg.base import load_doc, check_text_layer
doc = load_doc(pdf_path, 'A0001')
print('проверка текстового слоя:', check_text_layer(doc) or 'пройдена')
print('язык:', doc.lang, '| страниц:', doc.meta['n_pages'], '| символов текста:', len(doc.text))
print('абзацев:', len(doc.paragraphs), '| предложений:', len(doc.sentences),
      '| выключных формул:', len(doc.formulas), '| гиперссылок PDF:', len(doc.links))

проверка текстового слоя: пройдена
язык: en | страниц: 27 | символов текста: 48581
абзацев: 427 | предложений: 772 | выключных формул: 267 | гиперссылок PDF: 193


Абзацы — основная единица текста: на них ссылаются фрагменты модулей, и в графе они становятся
сегментами `moc:Paragraph` с номерами `p-0001`, `p-0002`, …

In [4]:
for i, p in enumerate(doc.paragraphs[:6]):
    print(f'p-{i + 1:04d}  стр. {p.page}:', ' '.join(doc.text[p.start:p.end].split())[:160])

p-0001  стр. 1: MAXIMUM PRINCIPLES AND PRINCIPAL EIGENVALUE FOR
p-0002  стр. 1: COUPLED LOCAL/NONLOCAL MODELS
p-0003  стр. 1: WILLIAN CINTRA1, JˆONATAS PERALTA1, JULIO D. ROSSI2, AND ANTONIO SU´AREZ3
p-0004  стр. 1: arXiv:2611.16131v1 [math.AP] 12 Nov 2026
p-0005  стр. 1: Abstract. In this paper, we analyze the validity of the maximum principle
p-0006  стр. 1: and the existence of a principal eigenvalue for a problem involving two regions,


Отметки шрифта (`doc.styled`) сохраняют то, что в PDF видно глазом и теряется в простом тексте:
полужирный заголовок блока «Theorem 2.3.», курсив термина и формулировки, математический шрифт.
На этих признаках строятся правила модулей.

In [5]:
from collections import Counter
print(Counter(s.kind for s in doc.styled))
for kind in ('bold', 'italic', 'math'):
    print(kind, '→', [doc.text[s.start:s.end] for s in doc.styled if s.kind == kind][:8])

Counter({'italic': 1166, 'math': 1039, 'bold': 41})
bold → ['MAXIMUM PRINCIPLES AND PRINCIPAL EIGENVALUE FOR', 'COUPLED LOCAL/NONLOCAL MODELS', 'Keywords:', 'Mathematics Subject Classification (2010):', 'Problem and Hypotheses.', 'Remark 1.1.', 'Remark 1.2.', 'Notations and Main results:']
italic → ['ℓ', 'nℓ', 'N', 'N ≥', 'ℓ∪Ωnℓ∪', '∂Ωℓ∩∂Ωnℓ', 'ℓ,', 'nℓ']
math → ['ℓ', 'nℓ', 'RN', 'N ≥', 'ℓ∪Ωnℓ∪', 'Γ', '∂Ωℓ∩∂Ωnℓ', 'ℓ,']


Выключные формулы `base` находит по положению строки на странице и математическому шрифту и
отделяет номер «(2.3)» у правого края. Текстовый слой PDF теряет структуру формулы (дроби, индексы,
пределы интегралов), поэтому `base` умеет восстанавливать LaTeX формулы по картинке моделью
распознавания (pix2tex, нужен GPU); в этом блокноте распознавание выключено.

In [6]:
numbered = [f for f in doc.formulas if f.number]
print(len(numbered), 'формул с номером из', len(doc.formulas))
for f in numbered[:6]:
    print(f'({f.number})  стр. {f.page}:', f.text[:90])

RECOGNIZE = False            # True — в Colab с GPU после !pip install pix2tex
if RECOGNIZE:
    from mathkg.base import recognize_formulas
    recognize_formulas(doc, pdf_path, limit=5)
    for f in doc.formulas[:5]:
        print(f.number, f.latex)

23 формул с номером из 267
(2.1)  стр. 6: Z L2h = − J(x −y)h(y)dy ≥0 in Ωnℓ.
(2.2)  стр. 8: ( Lu(x) + B(x)u(x) = 0 x ∈Ω, Bu(x) = 0 x ∈∂Ωℓ,
(2.3)  стр. 8: ( Lu(x) + B(x)u(x) = f(x) x ∈Ω, Bu(x) = g(x) x ∈∂Ωℓ,
(3.1)  стр. 9: n o u ∈X : u(x) > 0 for all x ∈Ωand Bu ≥0 on ∂Ωℓ P = and λp := sup inf x∈Ω ϕ∈P (L + B(x))ϕ
(3.2)  стр. 9: L2u(x) + B2(x)u(x) = L0u(x) + a(x)u(x) in Ωnℓ.
(3.3)  стр. 10: ( Lu(x) + B(x)u(x) −λpu(x) = φp(x) x ∈Ω, Bu(x) = 0 x ∈∂Ωℓ,


Статьи, свёрстанные в LaTeX с пакетом hyperref, содержат внутренние гиперссылки: ссылка
«Theorem 2.3» в тексте ведёт на цель `theorem.2.3`, «[12]» — на `cite.…`, «(2.1)» — на
`equation.2.1`. `base` сохраняет и текст ссылки, и её цель; модули ссылок и литературы получают
готовые связи без разбора текста.

In [7]:
print(Counter(l.target.split('.')[0] for l in doc.links))
for kind in ('theorem', 'equation', 'section', 'cite'):
    for l in [l for l in doc.links if l.start >= 0 and l.target.startswith(kind + '.')][:2]:
        print(f'{doc.text[l.start:l.end]!r:22}', '→', l.target)

Counter({'cite': 52, 'page': 44, 'theorem': 34, 'equation': 34, 'Item': 12, 'AMS': 11, 'section': 4, 'figure': 2})
'2.4)'                 → theorem.2.4
'3.4'                  → theorem.3.4
'(2.1)'                → equation.2.1
'(2.2)'                → equation.2.2
'2'                    → section.2
'3,'                   → section.3
'[3,'                  → cite.BCh
'4,'                   → cite.Bere


## 3. Модули извлечения и реестр инструментов

Каждый модуль — функция с декоратором `@tool`: она получает `Doc` и контекст с результатами модулей,
от которых зависит, и возвращает `Facts` — фрагменты (отрезки текста с классом онтологии) и связи
(тройки). `pipeline.setup()` регистрирует базовые версии десяти модулей; реестр знает зависимости и
запускает модули в нужном порядке.

In [8]:
from mathkg import pipeline, registry
pipeline.setup()
pd.DataFrame([dict(инструмент=n, модуль=s.module,
                   зависит_от=', '.join(registry.REGISTRY[r].module for r in s.requires) or '—',
                   классы=', '.join(s.produces)[:70])
              for n, s in registry.REGISTRY.items() if s.kind == 'extract'])

,инструмент,модуль,зависит_от,классы
0,extract_metadata,m01,—,"moc:Title, moc:Abstract, moc:KeywordList, scilibreco:Author, scilibrec"
1,extract_structure,m02,—,"moc:Section, moc:Subsection, moc:Introduction, moc:ConclusionSection,"
2,extract_statements,m03,—,"moc:Theorem, moc:Lemma, moc:Proposition, moc:Corollary, moc:Claim, moc"
3,extract_definitions,m04,—,"moc:Definition, moc:MathematicalConcept"
4,extract_formulas,m05,—,"moc:Equation, moc:Formula"
5,extract_bibliography,m06,—,"moc:Bibliography, moc:ReferenceEntry"
6,extract_proofs,m07,m03,"moc:Proof, moc:proves, moc:provedBy"
7,extract_references,m08,"m02, m03, m04, m06, m07","moc:refersToTheorem, moc:refersToFormula, moc:usesLemma, moc:citesBibl"
8,extract_concepts,m09,"m01, m04","moc:mentionsConcept, moc:mentions, moc:MathematicalStatement"
9,classify_topics,m10,m01,scilibreco:hasSubjectArea


In [9]:
results = registry.run_on_doc(doc)          # все модули на одной статье
pd.DataFrame([dict(инструмент=n, фрагментов=len(f.fragments), связей=len(f.relations), секунд=f.sec)
              for n, f in results.items()])

,инструмент,фрагментов,связей,секунд
0,extract_metadata,3,20,0.001
1,extract_structure,7,10,0.001
2,extract_statements,17,34,0.003
3,extract_definitions,5,25,0.003
4,extract_formulas,311,1428,0.019
5,extract_bibliography,21,20,0.001
6,extract_proofs,15,30,0.001
7,extract_references,0,93,0.006
8,extract_concepts,0,39,0.108
9,classify_topics,0,2,0.000


Модуль 3 нашёл утверждения статьи, модуль 7 — доказательства и связал каждое доказательство с
утверждением, которое оно доказывает (`moc:proves`). Модуль 7 зависит от модуля 3: связь строится
по его результатам из контекста.

In [10]:
txt = lambda f, n=110: ' '.join(doc.text[f.start:f.end].split())[:n]
st = {f.id: f for f in results['extract_statements'].fragments}
for r in results['extract_proofs'].relations:
    if r.pred == 'moc:proves':
        print(f'{r.subj} доказывает {st[r.obj].label}:', txt(st[r.obj], 75))

prf-0001 доказывает Theorem 2.3: Theorem 2.3. Suppose that (L + B, B, Ω) admits a positive supersolution h ∈
prf-0002 доказывает Corollary 2.4: Corollary 2.4. If (L + B, B, Ω) admits a positive strict supersolution h ∈X
prf-0003 доказывает Corollary 2.5: Corollary 2.5. Suppose that (L + B, B, Ω) admits a positive strict supersol
prf-0004 доказывает Corollary 2.6: Corollary 2.6. Suppose that (L + B, B, Ω) admits a positive strict supersol
prf-0005 доказывает Proposition 3.4: Proposition 3.4. If λ ∈R is a principal eigenvalue of (L + B, B, Ω), then
prf-0006 доказывает Theorem 3.5: Theorem 3.5. Assume that λp < m. Then the following statements are true. (i
prf-0007 доказывает Theorem 3.7: Theorem 3.7. Assume that λp < m. The following statements are equivalent: (
prf-0008 доказывает Theorem 3.8: Theorem 3.8. If there exists a measurable set A ⊂Ωnℓsuch that J(x −y) a(x) 
prf-0009 доказывает Proposition 4.1: Proposition 4.1. The equality λp = µp holds.
prf-0010 доказывает Proposition 4.2: P

Модуль 8 находит в тексте ссылки и превращает ссылки внутри доказательства в зависимости: если
доказательство следствия ссылается на теорему, появляется связь `moc:usesTheorem` или
`moc:usesResult`. В списке видна и ошибка базовых версий: доказательство prf-0002 «использует»
следствие 2.4, которое само доказывает. Модуль 7 закончил это доказательство на абзаце после знака □
(«We conclude this section with two consequences of Corollary 2.4»), и модуль 8 принял упоминание
в этом абзаце за ссылку из доказательства. Граница доказательства — работа варианта 7, отбрасывание
ссылки на доказываемое утверждение — варианта 8.

In [11]:
for r in results['extract_references'].relations:
    if r.pred.startswith('moc:uses'):
        print(r.subj, r.pred, '→', st[r.obj].label if r.obj in st else r.obj)

prf-0002 moc:usesTheorem → Theorem 2.3
prf-0002 moc:usesResult → Corollary 2.4
prf-0003 moc:usesResult → Corollary 2.4
prf-0007 moc:usesResult → Corollary 2.4
prf-0011 moc:usesResult → Proposition 4.2
prf-0012 moc:usesTheorem → Theorem 3.5
prf-0012 moc:usesResult → Proposition 4.4
prf-0013 moc:usesTheorem → Theorem 3.5
prf-0013 moc:usesTheorem → Theorem 2.3
prf-0015 moc:usesResult → Proposition 4.4
prf-0015 moc:usesLemma → Lemma 5.2


Модуль 4 находит определения и понятие, которое определение вводит. У понятия общий IRI
`mp:concept/…`, построенный из нормализованного имени: одно и то же понятие из разных статей и
разных наборов получает один IRI, и графы студентов соединяются через понятия. Базовая версия
модуля берёт термином имя в скобках после заголовка «Definition 2.1 (…)» или первый курсивный
отрезок определения. В этой статье у определений нет имён в скобках, а первый курсивный отрезок
часто оказывается служебными словами или куском формулы, поэтому для части определений понятие не
найдено или найдено неверно. Раздел 8 показывает, как студенческая версия модуля это исправляет.

In [12]:
dfs = results['extract_definitions']
concept = {r.subj: r.obj for r in dfs.relations if r.pred == 'moc:defines'}
for f in dfs.fragments:
    print(f.id, f.label or '(внутри абзаца)', '|', txt(f, 75))
    print('      понятие:', concept.get(f.id, '—'))

def-0001 (внутри абзаца) | In this case we say that u is positive.
      понятие: https://w3id.org/mocassinplus/resource/concept/positive
def-0002 Definition 2.1 | Definition 2.1. We say that u ∈X is a supersolution of (L + B, B, Ω) if
      понятие: —
def-0003 Definition 2.2 | Definition 2.2. We say that (L+B, B, Ω) satisfies the Strong Maximum Princi
      понятие: https://w3id.org/mocassinplus/resource/concept/say-that-we
def-0004 Definition 3.1 | Definition 3.1. We say that a value λ ∈R is an eigenvalue of the operator (
      понятие: https://w3id.org/mocassinplus/resource/concept/eigenvalue-is-operator
def-0005 Definition 5.1 | Definition 5.1. We say that u ∈H is a weak solution of (Pλ) if u|Ωℓand u|Ωn
      понятие: https://w3id.org/mocassinplus/resource/concept/are-if-n-solution-u-u-weak


У каждого фрагмента записано, как он получен: метод (`rule` — правило, `model` — классификатор,
`llm` — языковая модель), уверенность, страница и позиции в тексте. В графе эти сведения становятся
свойствами происхождения факта, и агент может ответить, откуда взят каждый факт.

In [13]:
from dataclasses import asdict
from mathkg.model import resolve
f = results['extract_statements'].fragments[0]
print('IRI фрагмента:', resolve(doc.doc_id, f.id))
asdict(f)

IRI фрагмента: https://w3id.org/mocassinplus/resource/doc/A0001/st-0001


{'id': 'st-0001',
 'cls': 'moc:Theorem',
 'start': 11858,
 'end': 12102,
 'method': 'rule',
 'confidence': 0.9,
 'number': '2.3',
 'label': 'Theorem 2.3',
 'page': 6,
 'attrs': {}}

## 4. Прогон по набору и проверка контракта

На своём наборе студент работает с папками: `build_corpus` прогоняет `base` по всем PDF из
`index.csv` и сохраняет `Doc` каждой статьи, `run_corpus` запускает модули и сохраняет результаты.
Оба вызова продолжают работу после обрыва сеанса Colab с того места, где остановились. `build_report`
проверяет контракт каждого модуля: классы и свойства есть в онтологии, позиции фрагментов не выходят
за текст, связи ссылаются на существующие фрагменты, время на статью не больше 2 с. Эту таблицу
руководитель показывает на сборках группы.

In [14]:
from mathkg.base import build_corpus
from mathkg.check import build_report
BASE, FACTS = os.path.join(WORK, 'base'), os.path.join(WORK, 'facts')
print('base:', build_corpus(os.path.join(WORK, 'index.csv'), PDF_DIR, BASE))
pipeline.run_corpus(BASE, FACTS)
report = pd.DataFrame(build_report(BASE, FACTS, ONTOLOGY))
report[['module', 'tool', 'author', 'passed', 'errors', 'fragments', 'relations', 'ms_per_doc']]

base: {'ok': 1}


,module,tool,author,passed,errors,fragments,relations,ms_per_doc
0,m01,extract_metadata,базовая версия,True,0,3,20,1.0
1,m02,extract_structure,базовая версия,True,0,7,10,1.0
2,m03,extract_statements,базовая версия,True,0,17,34,3.0
3,m04,extract_definitions,базовая версия,True,0,5,25,3.0
4,m05,extract_formulas,базовая версия,True,0,311,1428,19.0
5,m06,extract_bibliography,базовая версия,True,0,21,20,1.0
6,m07,extract_proofs,базовая версия,True,0,15,30,1.0
7,m08,extract_references,базовая версия,True,0,0,93,6.0
8,m09,extract_concepts,базовая версия,True,0,0,39,106.0
9,m10,classify_topics,базовая версия,True,0,0,2,0.0


## 5. Оценка по ручному эталону

Качество модуля измеряется только по ручному эталону: исходников статей нет, и правильный ответ
знает человек, который прочитал статью. Эталон — файл `gold/mNN.jsonl`, по строке на объект;
позиции не нужны, достаточно дословной цитаты начала фрагмента. Эталон перечисляет все объекты
своего класса в статье: иначе верные находки модуля засчитаются как ложные. Ниже эталон модуля 3
(m03) на этой статье — все пять теорем и лемма.

Цитату надёжнее копировать из текста `base` (`doc.text` или инструмент `find_in_article`): в этой
статье буква Ω набрана знаком ома (U+2126), и цитата с греческой Ω (U+03A9), набранная с клавиатуры,
в тексте не найдётся — `evaluate` перечислит её в `unresolved_quotes`.

In [15]:
from mathkg.evaluate import evaluate
gold = [
    {"doc_id": "A0001", "cls": "moc:Theorem", "number": "2.3", "quote": "Theorem 2.3. Suppose that (L + B, B, Ω) admit"},
    {"doc_id": "A0001", "cls": "moc:Theorem", "number": "3.5", "quote": "Theorem 3.5. Assume that λp < m. Then the fol"},
    {"doc_id": "A0001", "cls": "moc:Theorem", "number": "3.7", "quote": "Theorem 3.7. Assume that λp < m. The followin"},
    {"doc_id": "A0001", "cls": "moc:Theorem", "number": "3.8", "quote": "Theorem 3.8. If there exists a measurable set"},
    {"doc_id": "A0001", "cls": "moc:Theorem", "number": "5.3", "quote": "Theorem 5.3. If J is symmetric, then the equa"},
    {"doc_id": "A0001", "cls": "moc:Lemma", "number": "5.2", "quote": "Lemma 5.2. If J is symmetric, then B(x)ϕ(x)2d"},
]
GOLD = os.path.join(WORK, 'gold_m03.jsonl')
with open(GOLD, 'w', encoding='utf-8') as f:
    f.writelines(json.dumps(g, ensure_ascii=False) + '\n' for g in gold)
ev = evaluate('extract_statements', GOLD, BASE, FACTS)
print('пропущено:', len(ev['missed']), '| лишних:', len(ev['extra']), '| цитат не найдено:', len(ev['unresolved_quotes']))
pd.DataFrame(ev['by_key']).T

пропущено: 0 | лишних: 0 | цитат не найдено: 0


,precision,recall,f1,tp,pred,gold
moc:Lemma,1.0,1.0,1.0,1.0,1.0,1.0
moc:Theorem,1.0,1.0,1.0,5.0,5.0,5.0


## 6. Граф RDF и запросы SPARQL

`corpus_to_ntriples` переводит статьи и результаты модулей в тройки по общей модели данных
(онтологии MocassinPlus v2 и SciLibReco в папке `ontology/`). На практике граф загружается в
GraphDB, в именованный граф студента; здесь он читается библиотекой `rdflib` прямо в блокноте.

In [16]:
from mathkg.to_rdf import corpus_to_ntriples
import rdflib
NT = os.path.join(WORK, 'graph.nt')
n = corpus_to_ntriples(BASE, FACTS, NT, 'student-00')
g = rdflib.Graph()
g.parse(NT, format='nt')
print(n, 'троек записано,', len(g), 'прочитано rdflib')

PREFIXES = '''PREFIX moc: <https://w3id.org/mocassinplus/ontology#>
PREFIX scilibreco: <https://scilib.ai/ontology/scilibreco#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
'''
def sparql(q):
    res = g.query(PREFIXES + q)
    return pd.DataFrame([[str(x) for x in row] for row in res], columns=[str(v) for v in res.vars])

sparql('SELECT ?class (COUNT(?x) AS ?n) WHERE { ?x a ?class } GROUP BY ?class ORDER BY DESC(?n) LIMIT 12')

9418 троек записано, 9197 прочитано rdflib


,class,n
0,https://w3id.org/mocassinplus/ontology#Formula,578
1,https://w3id.org/mocassinplus/ontology#Paragraph,323
2,https://w3id.org/mocassinplus/ontology#Equation,97
3,https://w3id.org/mocassinplus/ontology#ReferenceEntry,20
4,https://w3id.org/mocassinplus/ontology#Proof,15
5,https://w3id.org/mocassinplus/ontology#ExtractionRun,8
6,https://w3id.org/mocassinplus/ontology#ExtractionTool,8
7,https://w3id.org/mocassinplus/ontology#Proposition,6
8,https://w3id.org/mocassinplus/ontology#Theorem,5
9,https://w3id.org/mocassinplus/ontology#Corollary,5


Запрос ниже находит утверждения статьи и их доказательства, второй — на какие утверждения
опираются доказательства (связи, которые построил модуль 8).

In [17]:
sparql('''SELECT ?label ?proof WHERE {
  ?s moc:hasLabel ?label ; moc:provedBy ?proof .
} ORDER BY ?label''').head(8)

,label,proof
0,Corollary 2.4,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0002
1,Corollary 2.5,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0003
2,Corollary 2.6,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0004
3,Corollary 4.3,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0011
4,Corollary 4.7,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0013
5,Lemma 5.2,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0014
6,Proposition 3.4,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0005
7,Proposition 4.1,https://w3id.org/mocassinplus/resource/doc/A0001/prf-0009


In [18]:
sparql('''SELECT ?proved ?relation ?used WHERE {
  ?p moc:proves ?t . ?t moc:hasLabel ?proved .
  ?p ?relation ?u . ?u moc:hasLabel ?used .
  FILTER(?relation IN (moc:usesTheorem, moc:usesLemma, moc:usesResult, moc:usesDefinition))
}''')

,proved,relation,used
0,Corollary 2.4,https://w3id.org/mocassinplus/ontology#usesTheorem,Theorem 2.3
1,Corollary 2.4,https://w3id.org/mocassinplus/ontology#usesResult,Corollary 2.4
2,Corollary 2.5,https://w3id.org/mocassinplus/ontology#usesResult,Corollary 2.4
3,Theorem 3.7,https://w3id.org/mocassinplus/ontology#usesResult,Corollary 2.4
4,Corollary 4.3,https://w3id.org/mocassinplus/ontology#usesResult,Proposition 4.2
5,Proposition 4.6,https://w3id.org/mocassinplus/ontology#usesTheorem,Theorem 3.5
6,Proposition 4.6,https://w3id.org/mocassinplus/ontology#usesResult,Proposition 4.4
7,Corollary 4.7,https://w3id.org/mocassinplus/ontology#usesTheorem,Theorem 2.3
8,Corollary 4.7,https://w3id.org/mocassinplus/ontology#usesTheorem,Theorem 3.5
9,Theorem 5.3,https://w3id.org/mocassinplus/ontology#usesResult,Proposition 4.4


## 7. Модули как инструменты агента

`registry.agent_tools` отдаёт все зарегистрированные инструменты в формате, который понимают
шаблоны чата моделей `transformers` и MCP: имя, описание (докстринг функции модуля) и JSON-схема
параметров. Агент выбирает инструмент по описанию, поэтому описание модуля пишется для читателя-
модели. Кроме десяти инструментов извлечения в реестре есть инструменты запросов к тексту статьи
`get_article_text` и `find_in_article`; на этапах 8–10 к ним добавятся запросы SPARQL и поиск.

In [19]:
from mathkg.base import read_doc
os.environ['KG_BASE_DIR'] = BASE              # папка base набора для инструментов запросов
tools = registry.agent_tools(lambda doc_id: read_doc(BASE, doc_id))
for t in tools:
    print(f"{t['function']['name']:22s}", t['function']['description'].split('\n')[0][:95])

extract_metadata       Метаданные статьи: заголовок, аннотация, ключевые слова, авторы и их e-mail, УДК, MSC,
extract_structure      Разделы статьи с номерами и названиями, их тип и состав (абзацы раздела), подписи к
extract_statements     Утверждения статьи с заголовком: теоремы, леммы, предложения, следствия, гипотезы.
extract_definitions    Определения (блоки и предложения внутри абзаца) и определяемые понятия: связь
extract_formulas       Формулы статьи: уравнения среди выключных формул, нормализованная запись, строчные
extract_bibliography   Список литературы: записи с номерами, DOI и годом; связь записи с цитируемой работой
extract_proofs         Доказательства статьи и связь каждого с утверждением, которое оно доказывает
extract_references     Ссылки внутри статьи на утверждения, определения, формулы, разделы и литературу;
extract_concepts       Упоминания понятий статьи в абзацах (moc:mentionsConcept) и именованные результаты
classify_topics        Разделы arXiv, к которым отно

In [20]:
t = next(t for t in tools if t['function']['name'] == 'extract_proofs')
print(json.dumps(t['function'], ensure_ascii=False, indent=1))
out = t['callable'](doc_id='A0001')          # так инструмент вызывает агент
print(out['n_fragments'], 'доказательств; первое:')
out['fragments'][0]

{
 "name": "extract_proofs",
 "description": "Доказательства статьи и связь каждого с утверждением, которое оно доказывает\n(moc:proves и обратная moc:provedBy).",
 "parameters": {
  "type": "object",
  "properties": {
   "doc_id": {
    "type": "string",
    "description": "идентификатор статьи набора, например A04571"
   }
  },
  "required": [
   "doc_id"
  ]
 }
}
15 доказательств; первое:


{'iri': 'https://w3id.org/mocassinplus/resource/doc/A0001/prf-0001',
 'cls': 'moc:Proof',
 'number': '2.3',
 'label': 'Proof',
 'method': 'rule',
 'confidence': 0.6,
 'text': 'Proof. First, we will show that h ≫0. Since h is positive, it follows that either h > 0 in Ωℓor h > 0 in Ωnℓ. We analyze these two possibilities:\n\nCase 1: Suppose that h > 0 in Ωℓand let M > 0 be a constant such that Jnℓ+ B1 + M > 0 in Ωℓ. Since h > '}

In [21]:
find = next(t for t in tools if t['function']['name'] == 'find_in_article')['callable']
find(doc_id='A0001', phrase='principal eigenvalue', limit=3)

[{'id': 'p-0001',
  'page': 1,
  'text': 'MAXIMUM PRINCIPLES AND PRINCIPAL EIGENVALUE FOR'},
 {'id': 'p-0006',
  'page': 1,
  'text': 'and the existence of a principal eigenvalue for a problem involving two regions,'},
 {'id': 'p-0012',
  'page': 1,
  'text': 'Keywords: Maximum principles; principal eigenvalue; local operator; nonlocal operator.'}]

## 8. Студенческий модуль заменяет базовую версию

Студент пишет свою версию модуля в файле `mNN_<название>.py` с декоратором `@tool` и тем же именем
инструмента, что у базовой версии; `pipeline.setup(student_modules=папка)` регистрирует базовые
версии, затем файлы из папки, и студенческая версия заменяет базовую везде: в прогоне, в графе и у
агента. Имя инструмента, зависимости и формат результата менять нельзя, внутри функции можно всё.

Пример ниже — версия модуля 4 (m04), которая берёт термин определения из оборотов «u is a
supersolution of …», «satisfies the Strong Maximum Principle if …»; если оборота нет, остаётся
термин базовой версии. Шаблон пустого модуля — `mathkg/modules_template/mNN_template.py`.

In [22]:
MY_MODULES = os.path.join(WORK, 'modules')
os.makedirs(MY_MODULES, exist_ok=True)
code = r'''
import re
from mathkg.common import concept_iri, normalize_name
from mathkg.model import Context, Doc, Facts
from mathkg.registry import tool
from mathkg.modules.m04_definitions import extract_definitions as base_version

TERM = re.compile(r"\b(?:is|are) (?:called |said to be )?(?:an? |the )?([a-z][a-z -]{2,40}?) (?:of|if)\b"
                  r"|satisfies the ([A-Za-z][A-Za-z -]{2,40}?) if\b")


@tool(name="extract_definitions", module="m04", produces=("moc:Definition", "moc:MathematicalConcept"),
      author="пример из блокнота")
def extract_definitions(doc: Doc, ctx: Context) -> Facts:
    """Определения статьи и понятия, которые они вводят. Термин берётся из оборота «X is a TERM of/if»
    или «satisfies the TERM if»; без такого оборота — первый курсивный отрезок определения."""
    facts = base_version(doc, ctx)
    base_concept = {r.subj: r.obj for r in facts.relations if r.pred == "moc:defines"}
    base_name = {r.subj: r.obj for r in facts.relations if r.pred == "scilibreco:hasName"}
    facts.relations = [r for r in facts.relations if r.pred == "moc:isPartOf"]   # понятия строятся заново
    for f in facts.fragments:
        m = TERM.search(" ".join(doc.text[f.start:f.end].split()))
        term = (m.group(1) or m.group(2)) if m else base_name.get(base_concept.get(f.id))
        if not term or not normalize_name(term):
            continue
        c = concept_iri(term)
        facts.add(c, "rdf:type", "moc:MathematicalConcept")
        facts.add(c, "scilibreco:hasName", term, literal=True)
        facts.add(c, "scilibreco:hasNormalizedName", normalize_name(term), literal=True)
        facts.add(f.id, "moc:defines", c, confidence=f.confidence)
        facts.add(doc.iri, "scilibreco:hasConcept", c, confidence=f.confidence)
    return facts
'''
with open(os.path.join(MY_MODULES, 'm04_definitions.py'), 'w', encoding='utf-8') as f:
    f.write(code)

def concepts(res):                     # определение -> имя понятия, которое оно вводит
    names = {r.subj: r.obj for r in res.relations if r.pred == 'scilibreco:hasName'}
    return {r.subj: names.get(r.obj) for r in res.relations if r.pred == 'moc:defines'}

before = concepts(results['extract_definitions'])
pipeline.setup(student_modules=MY_MODULES)
print('автор модуля m04:', registry.REGISTRY['extract_definitions'].author)
after = concepts(registry.run_on_doc(doc, ['extract_definitions'])['extract_definitions'])
pd.DataFrame([dict(определение=f.id, текст=txt(f, 60), базовая_версия=before.get(f.id, '—'),
                   версия_студента=after.get(f.id, '—'))
              for f in results['extract_definitions'].fragments])

автор модуля m04: пример из блокнота


,определение,текст,базовая_версия,версия_студента
0,def-0001,In this case we say that u is positive.,positive,positive
1,def-0002,Definition 2.1. We say that u ∈X is a supersolution of (L +,—,supersolution
2,def-0003,"Definition 2.2. We say that (L+B, B, Ω) satisfies the Strong",We say that,Strong Maximum Principle
3,def-0004,Definition 3.1. We say that a value λ ∈R is an eigenvalue of,is an eigenvalue of the operator,eigenvalue
4,def-0005,Definition 5.1. We say that u ∈H is a weak solution of (Pλ),if u|Ωℓand u|Ωnℓ are weak solutions of,weak solution


Пересчёт своего модуля на наборе — `run_corpus(..., redo=[имя])`; модули, которые от него
зависят (здесь — модули 8 и 9), пересчитываются вместе с ним.

In [23]:
stats = pipeline.run_corpus(BASE, FACTS, redo=['extract_definitions'])
pd.DataFrame({name: dict(s) for name, s in stats.items() if s}).T     # только пересчитанные модули

,docs,fragments,relations,ms,ms_per_doc
extract_definitions,1.0,5.0,30.0,3.0,3.0
extract_references,1.0,0.0,93.0,6.0,6.0
extract_concepts,1.0,0.0,115.0,71.0,71.0


## 9. Общий кэш моделей Hugging Face

Модули, которым нужна модель, берут её из `mathkg.models`: модель загружается один раз за сеанс,
и все модули получают одну её копию (`get_sentence_model`, `get_pipeline`, `get_llm`). Ниже
кодировщик `intfloat/multilingual-e5-base` превращает утверждения и определения статьи в векторы,
и вопрос на естественном языке находит ближайшие фрагменты. На этапах 9–10 тот же приём, индекс
FAISS и связи графа составляют поисковую систему. Первая загрузка модели в Colab занимает около
минуты.

In [24]:
USE_MODELS = True
if USE_MODELS:
    import numpy as np
    from mathkg.models import get_sentence_model
    enc = get_sentence_model('intfloat/multilingual-e5-base')
    frags = [(f.label or f.cls, txt(f, 400)) for name in ('extract_statements', 'extract_definitions')
             for f in results[name].fragments]
    E = enc.encode(['passage: ' + t for _, t in frags], normalize_embeddings=True)
    question = 'when does the strong maximum principle hold?'
    q = enc.encode(['query: ' + question], normalize_embeddings=True)[0]
    for k in np.argsort(-(E @ q))[:4]:
        print(round(float(E[k] @ q), 3), frags[k][0], '|', frags[k][1][:110])

0.866 Corollary 2.4 | Corollary 2.4. If (L + B, B, Ω) admits a positive strict supersolution h ∈X, then (L + B, B, Ω) satisfies the 
0.863 Definition 2.2 | Definition 2.2. We say that (L+B, B, Ω) satisfies the Strong Maximum Principle if every supersolution u ∈X \ {
0.845 Theorem 3.7 | Theorem 3.7. Assume that λp < m. The following statements are equivalent: (i) λp > 0. (ii) (L + B, B, Ω) admit
0.803 Proposition 4.4 | Proposition 4.4. The equality λp = µp holds.


## Что дальше

| Раздел блокнота | Этап практики |
|---|---|
| 1–2. Набор и base | этап 1 — свой набор из 1000 статей через base; этап 2 — текст, токены и ошибки base |
| 3, 5, 8. Модули, эталон, своя версия | этапы 3–6 — свой модуль: правила, модель Hugging Face, связи и общие IRI, языковая модель |
| 4. Прогон и проверка | сборки группы на встречах 5 и 7; этап 7 — прогон всей библиотеки на своём наборе |
| 6. Граф и SPARQL | этап 7 — граф RDF в GraphDB; этап 8 — запросы SPARQL и свой инструмент запросов |
| 9. Векторы | этап 9 — эмбеддинги и FAISS; этап 10 — поиск «вектор + граф» |
| 7. Инструменты агента | этап 11 — агент над инструментами группы; этап 12 — MCP-сервер группы |

Постановки десяти заданий — `ЗАДАНИЯ.md`, общая картина практики — `КОНСПЕКТ.md`.